## KDD: Recuento

Lee los artefactos de `data/artefactos/` y `data/final/` y verifica sus columnas

### Imports

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

---
### 1. Artefactos y tablas finales

In [ ]:
proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent

for base in (proyecto / "data" / "final", proyecto / "data" / "artefactos"):
    assert base.exists(), f"No encuentro {base}. Corre KDD/a_transformaciones y KDD/b_artefactos antes"
    print(f"== {base.relative_to(proyecto)} ==")
    for exito in sorted(base.rglob("_SUCCESS")):
        d = exito.parent
        archivos = list(d.rglob("*.parquet"))
        filas = sum(pq.ParquetFile(f).metadata.num_rows for f in archivos)
        esquema = pq.read_schema(archivos[0])
        nombres = ", ".join(esquema.names[:8]) + (" ..." if len(esquema.names) > 8 else "")
        print(f"  {str(d.relative_to(base)):32s} {filas:>10,} filas | {nombres}")

### 2. Recuento por técnica

| Artefacto | Parte | Uso |
|---|---|---|
| `features_proveedor`, `features_entidad` | II | KMeans++, DBSCAN/HDBSCAN, jerárquico (estandarizar `features_raw` o columnas) |
| `matriz_entidad_proveedor` (+ `indice_*`) | III, VI | ALS implícito, vecindario, SVD como embedder; hold-out con `anio_max` |
| `matriz_entidad_familia` | III | Recomendación entidad → rubro / contenido |
| `grafo_bipartito` | IV | PageRank (GraphFrames), HITS (hubs = `E|`, authorities = `P|`) |
| `grafo_coparticipacion` | V, VI | Girvan-Newman / espectral (subgrafo), BigCLAM, Node2Vec |
| `data/final/hechos` | todas | Grano entidad–proveedor–familia, partición anio/mes |
| `data/final/er_*` | VII | Auditoría de la resolución de entidades y encuadre ético |

---
### 3. Uso de la data

```python
hechos  = spark.read.parquet("data/final/hechos")                          # grano entidad–proveedor–ítem
aristas = spark.read.parquet("data/artefactos/grafo_bipartito/aristas")    # columnas src/dst para GraphFrames
```